# Importing the packages and data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from texttable import Texttable
import latextable

In [3]:
import sys
sys.path.insert(1, '../sar_dirichlet')
import dirichlet_regression

In [4]:
from func_test import cos_similarity, create_features_matrices, rmse_aitchison, cosine_similarity_aitchison, r2_aitchison_adjusted

In [6]:
import sys
sys.path.append("../../coral_mapping/")
import segments_regressor
import coral_map_tests

# Loading Maupiti data

In [7]:
X = np.load('../../smote_compositional/data/maupiti_X_for_smote.npy')
Y = np.load('../../smote_compositional/data/maupiti_y_for_smote.npy')
segments = np.load('../../smote_compositional/data/maupiti_segments_for_smote.npy')
indices_segments_keep = np.load('../../smote_compositional/data/maupiti_indices_segments_for_smote.npy')

In [8]:
boundaries = coral_map_tests.find_boundaries(segments)
adjacent_segments,_ = segments_regressor.find_adjacent_segments(segments, boundaries)

In [9]:
n_features = 16
n_classes = 4
n_samples = X.shape[0]

In [10]:
Y_star = (Y*(n_samples-1)+1/n_classes)/n_samples

In [11]:
# Z only one intercept
Z = np.ones((n_samples,1))
gamma_0 = [0.]

In [12]:
X = StandardScaler().fit(X).transform(X)

In [13]:
# Z a copy of X
Z = np.copy(X)
gamma_0 = np.zeros(n_features)

In [14]:
W = np.zeros((np.max(segments)+1,np.max(segments)+1))
for i in range(np.max(segments)+1):
    W[i,adjacent_segments[i]]=1

In [15]:
W = W[indices_segments_keep][:,indices_segments_keep]

In [16]:
Wsum = W.sum(axis=1)[:,None]
Wsum[Wsum==0] = 1

In [17]:
Wsum = W.sum(axis=1)[:,None]
Wsum[Wsum==0] = 1
W = W/Wsum

In [18]:
np.shape(W)

(2301, 2301)

In [19]:
segments_ind, segments_count = np.unique(segments,return_counts=True)

In [20]:
size_segments = segments_count[indices_segments_keep]

In [21]:
# segments that have no neighbours
np.where(~W.any(axis=1))[0]

array([   3,    4,    5,    6,    7,    8,    9,   10,   11,   12,   15,
         20,   55,   94,  136,  209,  218,  226,  231,  238,  263,  267,
        287,  291,  312,  322,  346,  361,  375,  442,  542,  555,  686,
        884, 1027, 1040, 1081, 1239, 1298, 1335, 1624, 1680, 1879, 1984],
      dtype=int64)

In [22]:
# we set to 1 on each column for the rows where the segments have no neighbours
for i in np.where(~W.any(axis=1))[0]:
    W[i] += 1

In [23]:
X = np.array(pd.read_csv('Data Dirichlet/maupiti_X.csv', header=None))
Y_star = np.array(pd.read_csv('Data Dirichlet/maupiti_Y.csv', header=None))
W = np.array(pd.read_csv('Data Dirichlet/maupiti_W.csv', header=None))

In [24]:
n_features = 16
n_classes = 4
n_samples = X.shape[0]
n = n_samples
#Z = np.copy(X)
#gamma_0 = np.zeros(n_features)

In [25]:
# We only keep the gamma with p-value < 0.2 ; then, we run the model and only keep the significant gamma (we delete gamma 2 and gamma 15 after this round)
Z = np.copy(X)[:, [3, 5, 8, 9, 10, 12, 13]]
gamma_0 = np.zeros(Z.shape[1])

### Not spatial

In [26]:
%%time
dirichRegressor_ns = dirichlet_regression.dirichletRegressor()
dirichRegressor_ns.fit(X, Y_star, parametrization='alternative', gamma_0=gamma_0, Z=Z)

Optimization terminated successfully.
Wall time: 300 ms


In [27]:
len((dirichRegressor_ns.beta).flatten())+len(gamma_0)

58

In [28]:
min_thresh = np.min(Y_star)

In [29]:
min_thresh

0.0001086484137331

In [30]:
Y_pred = np.copy(dirichRegressor_ns.mu)
Y_pred[Y_pred<min_thresh] = min_thresh

In [33]:
k = len(dirichRegressor_ns.beta.flatten()) + len(dirichRegressor_ns.gamma.flatten())

print('With Maximum Likelihood')
print('R2 aitchison:', r2_aitchison_adjusted(Y_star,Y_pred,k))
print('R2 adjusted : ', 1 - (1 - r2_score(Y_star,Y_pred)) * (n-1)/(n-k))
print('RMSE:',mean_squared_error(Y_star,Y_pred,squared=False))
print('Cross-entropy:',1/n_samples*np.sum(Y_star*np.log(Y_pred)))
print('AIC:',-2*dirichlet_regression.dirichlet_loglikelihood(Y_pred,dirichRegressor_ns.phi,Y_star)+2*58)
print('Cos similarity aitchison:', cosine_similarity_aitchison(Y_star,Y_pred))
print('Cos similarity:',cos_similarity(Y_star,Y_pred))
print('RMSE_A:', rmse_aitchison(Y_star,Y_pred))

With Maximum Likelihood
R2 aitchison: 0.008178403513942833
R2 adjusted :  0.24136057125504173
RMSE: 0.2974603427427088
Cross-entropy: -0.8189964972198622
AIC: -73145.07140275175
Cos similarity aitchison: 0.7550523701394194
Cos similarity: 0.78464345445402
RMSE_A: 6.908575909862691


In [32]:
print('Euclidian distance')
k = len(dirichRegressor_ns.beta.flatten()) + len(dirichRegressor_ns.gamma.flatten())
print('R2 adjusted : ', 1 - (1 - r2_score(Y_star,Y_pred)) * (n-1)/(n-k))
print('Cos similarity:',cos_similarity(Y_star,Y_pred))

Euclidian distance
R2 adjusted :  0.24136057125504173
Cos similarity: 0.78464345445402


In [33]:
%%time
dirichRegressor_ns_ce = dirichlet_regression.dirichletRegressor()
dirichRegressor_ns_ce.fit(X, Y_star, parametrization='alternative', gamma_0=gamma_0, Z=Z, loss='crossentropy')

Optimization terminated successfully.
Wall time: 443 ms


In [34]:
len((dirichRegressor_ns_ce.beta).flatten())

51

In [35]:
Y_pred = np.copy(dirichRegressor_ns_ce.mu)
Y_pred[Y_pred<min_thresh] = min_thresh

In [36]:
print('With Cross-entropy')
print('R2:',r2_aitchison_adjusted(Y_star,Y_pred,51))
print('RMSE:',mean_squared_error(Y_star,Y_pred,squared=False))
print('Cross-entropy:',1/n_samples*np.sum(Y_star*np.log(Y_pred)))
print('Cos similarity:',cosine_similarity_aitchison(Y_star,Y_pred))
print('RMSE_A:', rmse_aitchison(Y_star,Y_pred))

With Cross-entropy
R2: 0.3116793395866687
RMSE: 0.22088003151351088
Cross-entropy: -0.45192713292429904
Cos similarity: 0.7246995612414778
RMSE_A: 5.540200125411053


In [37]:
print('Euclidian distance')
k = len(dirichRegressor_ns_ce.beta.flatten())
print('R2 adjusted : ', 1 - (1 - r2_score(Y_star,Y_pred)) * (n-1)/(n-k))
print('Cos similarity:',cos_similarity(Y_star,Y_pred))

Euclidian distance
R2 adjusted :  0.6274094596963838
Cos similarity: 0.8682633257520268


In [46]:
%%time
dirichRegressor_ns_ce_sizes = dirichlet_regression.dirichletRegressor()
dirichRegressor_ns_ce_sizes.fit(X, Y_star, parametrization='alternative', gamma_0=gamma_0, Z=Z, loss='crossentropy',size_samples=size_segments)

Optimization terminated successfully.
Wall time: 8.82 s


In [47]:
Y_pred = np.copy(dirichRegressor_ns_ce_sizes.mu)
Y_pred[Y_pred<min_thresh] = min_thresh

In [48]:
print('With Cross-entropy')
print('R2:',r2_aitchison_adjusted(Y_star,Y_pred,51))
print('RMSE:',mean_squared_error(Y_star,Y_pred,squared=False))
print('Cross-entropy:',1/n_samples*np.sum(Y_star*np.log(Y_pred)))
print('Cos similarity:',cosine_similarity_aitchison(Y_star,Y_pred))
print('RMSE_A:', rmse_aitchison(Y_star,Y_pred))

With Cross-entropy
R2: 0.29738294183683045
RMSE: 0.22299035818572582
Cross-entropy: -0.4554556370884759
Cos similarity: 0.7171673261667656
RMSE_A: 5.622039184167409


In [49]:
ll_ns = dirichlet_regression.dirichlet_loglikelihood(dirichRegressor_ns.mu,dirichRegressor_ns.phi,Y_star)
ll_ns

36948.9283051442

### Spatial

In [38]:
rho_temp = 0.9295644055153885
gamma_temp = np.array([ 0.03550203,  0.22873115,  0.2348769 , -0.24725498,  0.0297459 ,
       -0.12387438,  0.0533362 ])
beta_temp = np.array([[-1.30732174e-02,  5.54450911e-02,  2.50441943e-02],
       [ 4.51646419e-02, -6.84721075e-02,  5.40545642e-02],
       [-9.24849384e-02, -2.82555556e-02,  4.15198536e-02],
       [-3.30465770e-02, -5.03997269e-02, -3.48399409e-03],
       [-3.29974651e-02, -2.55951957e-02,  9.01467420e-03],
       [-9.82800764e-02,  8.82103924e-02, -3.29986779e-02],
       [ 1.22331237e-02, -7.88034375e-02,  1.16573352e-01],
       [-9.31369329e-03,  7.89527116e-02, -1.32855754e-01],
       [-9.04904229e-03,  5.11491110e-02, -1.17947897e-01],
       [ 2.82139348e-02,  2.10323407e-02, -3.09512772e-02],
       [ 7.34921257e-02,  1.04389511e-01, -1.80813936e-01],
       [-3.11123377e-02, -1.16903114e-02, -3.51178260e-02],
       [-1.69104643e-02, -4.10018064e-02, -1.89377146e-02],
       [-1.98223856e-02, -7.68195133e-02,  9.74930533e-03],
       [-2.53497032e-02,  1.79265837e-02, -1.60693840e-04],
       [-7.80293785e-03, -3.11162412e-02,  2.98239958e-02],
       [ 1.33852813e-02,  1.73186726e-02, -3.15959322e-02]])

In [39]:
%%time
dirichRegressor_s = dirichlet_regression.dirichletRegressor(spatial=True,maxfun=10000)
dirichRegressor_s.fit(X, Y_star, W=W, parametrization='alternative', rho_0=rho_temp, gamma_0=gamma_temp, beta_0=beta_temp, Z=Z)

CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Wall time: 30.3 s


In [48]:
Y_pred = np.copy(dirichRegressor_s.mu)
Y_pred[Y_pred<min_thresh] = min_thresh

In [41]:
print('With Maximum Likelihood (spatial)')
print('R2:', r2_aitchison_adjusted(Y_star,Y_pred,58))
print('RMSE:',mean_squared_error(Y_star,Y_pred,squared=False))
print('Cross-entropy:',1/n_samples*np.sum(Y_star*np.log(Y_pred)))
print('AIC:',-2*dirichlet_regression.dirichlet_loglikelihood(Y_pred,dirichRegressor_s.phi,Y_star)+2*53)
print('Cos similarity:', cosine_similarity_aitchison(Y_star,Y_pred))
print('RMSE_A:', rmse_aitchison(Y_star,Y_pred))

With Maximum Likelihood (spatial)
R2: 0.07295272363136274
RMSE: 0.2609009092387607
Cross-entropy: -0.6749557588319154
AIC: -74158.44066547698
Cos similarity: 0.8427429655153992
RMSE_A: 6.733934547317262


In [49]:
print('Euclidian distance')
k = len(dirichRegressor_s.beta.flatten()) + len(dirichRegressor_s.gamma.flatten()) + 1
print('R2 adjusted : ', 1 - (1 - r2_score(Y_star,Y_pred)) * (n-1)/(n-k))
print('Cos similarity:',cos_similarity(Y_star,Y_pred))

Euclidian distance
R2 adjusted :  0.4253053119407647
Cos similarity: 0.8481295450652601


In [30]:
-74158.44059821645 - 2*53 + 2*59

-74146.44059821645

In [107]:
ll_s = dirichlet_regression.dirichlet_loglikelihood(dirichRegressor_s.mu,dirichRegressor_s.phi,Y_star)
ll_s

37132.22029910822

In [44]:
%%time
dirichRegressor_s_ce = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor_s_ce.fit(X, Y_star, W=W, loss='crossentropy', 
                         rho_0 = 0.9039145221857615, beta_0 = dirichRegressor_ns_ce.beta)

Desired error not necessarily achieved due to precision loss.
Wall time: 4min 16s


In [45]:
Y_pred = np.copy(dirichRegressor_s_ce.mu)
Y_pred[Y_pred<min_thresh] = min_thresh

In [46]:
print('With Cross-entropy (spatial)')
print('R2:',r2_aitchison_adjusted(Y_star,Y_pred,52))
print('RMSE:',mean_squared_error(Y_star,Y_pred,squared=False))
print('Cross-entropy:',1/n_samples*np.sum(Y_star*np.log(Y_pred)))
print('Cos similarity:',cosine_similarity_aitchison(Y_star,Y_pred))
print('RMSE_A:', rmse_aitchison(Y_star,Y_pred))

With Cross-entropy (spatial)
R2: 0.47318793817422244
RMSE: 0.15867390432751072
Cross-entropy: -0.30437539780550166
Cos similarity: 0.7854829557847132
RMSE_A: 4.66614797701017


In [47]:
print('Euclidian distance')
k = len(dirichRegressor_s_ce.beta.flatten()) + 1
print('R2 adjusted : ', 1 - (1 - r2_score(Y_star,Y_pred)) * (n-1)/(n-k))
print('Cos similarity:',cos_similarity(Y_star,Y_pred))

Euclidian distance
R2 adjusted :  0.8176984392875684
Cos similarity: 0.9243347229546406


In [59]:
dirichRegressor_s_ce.rho

0.9039145221857615